# AuraGateway harness materializer input v3

Attach exactly one `ag-harness-be1bfad-v1-input` dataset. The producer accepts either the exact ZIP file or Kaggle's exact expanded source-tree representation, verifies the frozen identity, and emits one direct archive-free output.

Use Accelerator None, Internet Off, no secrets, and Save Version -> Save & Run All.


In [1]:
from __future__ import annotations

import hashlib
import json
import shutil
import stat
import zipfile
from pathlib import Path, PurePosixPath

NOTEBOOK_NAME = "ag-harness-materializer-input-v3"
DATASET_NAME = "ag-harness-be1bfad-v1-input"
DATASET_OWNER = "kabomolefe"

INPUT_ROOT = Path("/kaggle/input").resolve()
WORK_ROOT = Path("/kaggle/working").resolve()

EXPECTED_ARCHIVE_NAME = "ag-harness-be1bfad-v1.zip"
EXPECTED_ARCHIVE_SHA256 = "741629c9c1e39b02b14c16001ac3c7f96ebe6fb72670c47bfc22af31b4182c37"
EXPECTED_SOURCE_COMMIT = "be1bfadd8a8aa3f0a2f6143d6a73f082f1090c50"
EXPECTED_DIRECTORY_SHA256 = "4a371c80aef605c4f1ab5617c21ce43bd0939ad449ffcbcadab656878d785a2e"
EXPECTED_FILE_COUNT = 953
EXPECTED_TOTAL_BYTES = 8879194

OUTPUT_DIRECTORY_NAME = "auragateway_qualification_harness_be1bfad_v1"
OUTPUT_ROOT = WORK_ROOT / OUTPUT_DIRECTORY_NAME
RECEIPT_NAME = "ag_harness_materialization_receipt_v3.json"
RECEIPT_PATH = WORK_ROOT / RECEIPT_NAME
STAGING_ROOT = WORK_ROOT / ".ag_harness_materializer_staging"

MAXIMUM_FILES = 5000
MAXIMUM_TOTAL_BYTES = 100 * 1024 * 1024

ARCHIVE_SUFFIXES = (
    ".zip",
    ".tar",
    ".tgz",
    ".gz",
    ".bz2",
    ".xz",
    ".7z",
    ".whl",
)

REQUIRED_RELATIVE_PATHS = (
    "pyproject.toml",
    "README.md",
    "src/auragateway/local_abc/contracts.py",
    (
        "src/auragateway/local_abc/"
        "full_abc_local_environment_qualification_execution.py"
    ),
    (
        "src/auragateway/local_abc/"
        "full_abc_local_environment_qualification_execution_contracts.py"
    ),
    (
        "src/auragateway/local_abc/"
        "full_abc_local_environment_qualification_execution_"
        "authorization_issuance.py"
    ),
    (
        "src/auragateway/local_abc/"
        "full_abc_local_environment_qualification_kaggle_launcher.py"
    ),
    (
        "src/auragateway/local_abc/"
        "full_abc_local_environment_qualification_kaggle_runtime_adapter.py"
    ),
)


def canonical_json(payload: object) -> str:
    return json.dumps(
        payload,
        ensure_ascii=True,
        separators=(",", ":"),
        sort_keys=True,
    )


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


def resolve_dataset_root() -> Path:
    candidates = (
        INPUT_ROOT / DATASET_NAME,
        INPUT_ROOT / "datasets" / DATASET_OWNER / DATASET_NAME,
    )
    observed = tuple(
        candidate.resolve()
        for candidate in candidates
        if candidate.is_dir() and not candidate.is_symlink()
    )

    if len(observed) != 1:
        raise RuntimeError(
            "expected exactly one attached harness input dataset "
            f"but observed {len(observed)}"
        )

    dataset_root = observed[0]

    if INPUT_ROOT not in dataset_root.parents:
        raise RuntimeError("harness input dataset escaped /kaggle/input")

    return dataset_root


def inspect_directory(
    root: Path,
    *,
    reject_archives: bool,
) -> tuple[list[dict[str, object]], int]:
    entries: list[dict[str, object]] = []
    observed_total_bytes = 0

    for path in sorted(root.rglob("*"), key=lambda item: item.as_posix()):
        if path.is_symlink():
            raise RuntimeError("harness source contains a symbolic link")

        metadata = path.stat()

        if path.is_dir():
            continue

        if not stat.S_ISREG(metadata.st_mode):
            raise RuntimeError("harness source contains a non-regular member")

        relative_path = path.relative_to(root).as_posix()

        if reject_archives and relative_path.lower().endswith(
            ARCHIVE_SUFFIXES
        ):
            raise RuntimeError("harness source contains a nested archive")

        observed_total_bytes += metadata.st_size

        if observed_total_bytes > MAXIMUM_TOTAL_BYTES:
            raise RuntimeError("harness source exceeds the byte budget")

        entries.append(
            {
                "path": relative_path,
                "sha256": file_sha256(path),
                "size_bytes": metadata.st_size,
            }
        )

        if len(entries) > MAXIMUM_FILES:
            raise RuntimeError("harness source exceeds the file-count budget")

    if not entries:
        raise RuntimeError("harness source is empty")

    return entries, observed_total_bytes


def directory_identity(entries: list[dict[str, object]]) -> str:
    payload = json.dumps(
        entries,
        sort_keys=True,
        separators=(",", ":"),
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def validate_expected_tree(root: Path) -> tuple[list[dict[str, object]], int]:
    inventory, observed_total_bytes = inspect_directory(
        root,
        reject_archives=True,
    )
    observed_directory_sha256 = directory_identity(inventory)

    if len(inventory) != EXPECTED_FILE_COUNT:
        raise RuntimeError("harness source file count drifted")

    if observed_total_bytes != EXPECTED_TOTAL_BYTES:
        raise RuntimeError("harness source total bytes drifted")

    if observed_directory_sha256 != EXPECTED_DIRECTORY_SHA256:
        raise RuntimeError("harness source directory identity drifted")

    missing_required = tuple(
        relative_path
        for relative_path in REQUIRED_RELATIVE_PATHS
        if not (root / relative_path).is_file()
    )

    if missing_required:
        raise RuntimeError("harness source required file set is incomplete")

    return inventory, observed_total_bytes


def validated_member_path(member_name: str) -> PurePosixPath:
    member_path = PurePosixPath(member_name)

    if member_path.is_absolute() or ".." in member_path.parts:
        raise RuntimeError("harness archive contains an unsafe path")

    if not member_path.parts:
        raise RuntimeError("harness archive contains an empty path")

    return member_path


def extract_archive(archive_path: Path) -> None:
    with zipfile.ZipFile(archive_path) as archive:
        members = archive.infolist()
        regular_file_count = 0
        declared_total_bytes = 0

        for member in members:
            member_path = validated_member_path(member.filename)

            if member.flag_bits & 0x1:
                raise RuntimeError(
                    "harness archive contains an encrypted member"
                )

            unix_mode = member.external_attr >> 16
            member_type = stat.S_IFMT(unix_mode)

            if member.is_dir():
                continue

            if member_type not in (0, stat.S_IFREG):
                raise RuntimeError(
                    "harness archive contains a non-regular member"
                )

            if member_path.as_posix().lower().endswith(
                ARCHIVE_SUFFIXES
            ):
                raise RuntimeError(
                    "harness archive contains a nested archive"
                )

            regular_file_count += 1
            declared_total_bytes += member.file_size

            if regular_file_count > MAXIMUM_FILES:
                raise RuntimeError(
                    "harness archive exceeds the file-count budget"
                )

            if declared_total_bytes > MAXIMUM_TOTAL_BYTES:
                raise RuntimeError(
                    "harness archive exceeds the byte budget"
                )

        for member in members:
            member_path = validated_member_path(member.filename)
            destination = STAGING_ROOT.joinpath(*member_path.parts)

            if member.is_dir():
                destination.mkdir(parents=True, exist_ok=True)
                continue

            destination.parent.mkdir(parents=True, exist_ok=True)

            with archive.open(member, "r") as source:
                with destination.open("xb") as target:
                    shutil.copyfileobj(
                        source,
                        target,
                        length=1024 * 1024,
                    )


def copy_expanded_tree(source_root: Path) -> None:
    validate_expected_tree(source_root)

    for source_path in sorted(
        source_root.rglob("*"),
        key=lambda item: item.as_posix(),
    ):
        if source_path.is_symlink():
            raise RuntimeError("expanded harness contains a symbolic link")

        relative_path = source_path.relative_to(source_root)
        destination = STAGING_ROOT / relative_path

        if source_path.is_dir():
            destination.mkdir(parents=True, exist_ok=True)
            continue

        metadata = source_path.stat()

        if not stat.S_ISREG(metadata.st_mode):
            raise RuntimeError(
                "expanded harness contains a non-regular member"
            )

        destination.parent.mkdir(parents=True, exist_ok=True)

        with source_path.open("rb") as source:
            with destination.open("xb") as target:
                shutil.copyfileobj(
                    source,
                    target,
                    length=1024 * 1024,
                )


def materialize(dataset_root: Path) -> str:
    if OUTPUT_ROOT.exists():
        raise RuntimeError("materialized harness output already exists")

    if RECEIPT_PATH.exists():
        raise RuntimeError("materialization receipt already exists")

    if STAGING_ROOT.exists():
        raise RuntimeError("materializer staging directory already exists")

    archive_path = dataset_root / EXPECTED_ARCHIVE_NAME
    direct_tree_present = all(
        (dataset_root / relative_path).is_file()
        for relative_path in REQUIRED_RELATIVE_PATHS
    )
    archive_present = archive_path.is_file() and not archive_path.is_symlink()

    if archive_present and direct_tree_present:
        raise RuntimeError("harness input shape is ambiguous")

    if not archive_present and not direct_tree_present:
        raise RuntimeError(
            "harness dataset contains neither the exact archive "
            "nor the exact expanded source tree"
        )

    STAGING_ROOT.mkdir(parents=False)

    try:
        if archive_present:
            observed_archive_sha256 = file_sha256(archive_path)

            if observed_archive_sha256 != EXPECTED_ARCHIVE_SHA256:
                raise RuntimeError("attached harness archive identity drifted")

            extract_archive(archive_path)
            input_mode = "exact_archive"
        else:
            copy_expanded_tree(dataset_root)
            input_mode = "expanded_dataset_tree"

        validate_expected_tree(STAGING_ROOT)
        STAGING_ROOT.replace(OUTPUT_ROOT)
    except Exception:
        shutil.rmtree(STAGING_ROOT, ignore_errors=True)
        raise

    return input_mode


if len(NOTEBOOK_NAME) > 50:
    raise RuntimeError("Kaggle notebook name exceeds 50 characters")

dataset_root = resolve_dataset_root()
input_mode = materialize(dataset_root)
inventory, observed_total_bytes = validate_expected_tree(OUTPUT_ROOT)
observed_directory_sha256 = directory_identity(inventory)

receipt = {
    "schema_version": "1.0.0",
    "status": "HARNESS_MATERIALIZED",
    "producer_notebook_name": NOTEBOOK_NAME,
    "source_commit": EXPECTED_SOURCE_COMMIT,
    "input_dataset_name": DATASET_NAME,
    "input_mode": input_mode,
    "archive_filename": EXPECTED_ARCHIVE_NAME,
    "archive_sha256": EXPECTED_ARCHIVE_SHA256,
    "output_directory": OUTPUT_DIRECTORY_NAME,
    "directory_sha256": observed_directory_sha256,
    "file_count": len(inventory),
    "total_bytes": observed_total_bytes,
    "nested_archives_present": False,
    "symlinks_present": False,
    "network_access_performed": False,
    "gpu_execution_performed": False,
    "model_requests_performed": 0,
    "benchmark_trajectory_requests_performed": 0,
}

RECEIPT_PATH.write_text(
    canonical_json(receipt),
    encoding="utf-8",
)

print("status=HARNESS_MATERIALIZED")
print(f"input_dataset_root={dataset_root}")
print(f"input_mode={input_mode}")
print(f"output_directory={OUTPUT_ROOT}")
print(f"file_count={len(inventory)}")
print(f"total_bytes={observed_total_bytes}")
print(f"directory_sha256={observed_directory_sha256}")
print("nested_archives_present=false")
print("symlinks_present=false")
print("gpu_execution_performed=false")
print("model_requests_performed=0")
print("save_this_notebook_output=true")


status=HARNESS_MATERIALIZED
input_dataset_root=/kaggle/input/datasets/kabomolefe/ag-harness-be1bfad-v1-input
input_mode=expanded_dataset_tree
output_directory=/kaggle/working/auragateway_qualification_harness_be1bfad_v1
file_count=953
total_bytes=8879194
directory_sha256=4a371c80aef605c4f1ab5617c21ce43bd0939ad449ffcbcadab656878d785a2e
nested_archives_present=false
symlinks_present=false
gpu_execution_performed=false
model_requests_performed=0
save_this_notebook_output=true
